# **PyTorch Fundamentals for AI**

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# 📖 TABLE OF CONTENTS

- [Setup](#Setup)
- [PyTorch Tensors: Creation and Basics](#PyTorch-Tensors:-Creation-and-Basics)
  - [Exercises](#Exercises)
    - [Import torch](#Import-torch)
    - [Create a tensor from the list `[1.0, 2.0, 3.0, 4.0]` and store it in `tensor`.](#Create-a-tensor-from-the-list-`[1.0,-2.0,-3.0,-4.0]`-and-store-it-in-`tensor`.)
    - [Create a 2x3 tensor of zeros.](#Create-a-2x3-tensor-of-zeros.)
    - [Create a 3x3 tensor with random values from a normal distribution.](#Create-a-3x3-tensor-with-random-values-from-a-normal-distribution.)
    - [Print the shape, dtype, and device of the `tensor` variable.](#Print-the-shape,-dtype,-and-device-of-the-`tensor`-variable.)
- [PyTorch: Tensor Operations](#PyTorch:-Tensor-Operations)
  - [Exercises](#Exercises)
    - [Import torch and create tensor `[2.0, 4.0, 6.0, 8.0]`.](#Import-torch-and-create-tensor-`[2.0,-4.0,-6.0,-8.0]`.)
    - [Square each element and store in `squared`.](#Square-each-element-and-store-in-`squared`.)
    - [Create a 3x4 random tensor, then reshape it to 4x3 using `.view()`.](#Create-a-3x4-random-tensor,-then-reshape-it-to-4x3-using-`.view()`.)
    - [Create two tensors of shape (2,3) and (3,4), then multiply them using `@`.](#Create-two-tensors-of-shape-(2,3)-and-(3,4),-then-multiply-them-using-`@`.)
    - [Compute the mean and sum of tensor `C`.](#Compute-the-mean-and-sum-of-tensor-`C`.)
- [PyTorch: Autograd Basics](#PyTorch:-Autograd-Basics)
  - [Exercises](#Exercises)
    - [Import torch and create a tensor with value 3.0 and `requires_grad=True`.](#Import-torch-and-create-a-tensor-with-value-3.0-and-`requires_grad=True`.)
    - [Compute `y = x**2 + 2x + 1`.](#Compute-`y-=-x**2-+-2x-+-1`.)
    - [Call `backward()` on `y` to compute gradients.](#Call-`backward()`-on-`y`-to-compute-gradients.)
    - [Print `x.grad` to see the gradient dy/dx.](#Print-`x.grad`-to-see-the-gradient-dy/dx.)
    - [Create two tensors `a=2.0` and `b=3.0` with `requires_grad=True`, compute `c = a*b + a**2`, call backward, and print both gradients.](#Create-two-tensors-`a=2.0`-and-`b=3.0`-with-`requires_grad=True`,-compute-`c-=-a*b-+-a**2`,-call-backward,-and-print-both-gradients.)
- [PyTorch: Training Loop Pattern](#PyTorch:-Training-Loop-Pattern)
  - [Exercises](#Exercises)
    - [Import torch and create a weight tensor `W` with value 0.5 and `requires_grad=True`.](#Import-torch-and-create-a-weight-tensor-`W`-with-value-0.5-and-`requires_grad=True`.)
    - [Create input `x=2.0` and target `y_target=10.0` (no requires_grad needed for these).](#Create-input-`x=2.0`-and-target-`y_target=10.0`-(no-requires_grad-needed-for-these).)
    - [Set `learning_rate=0.01`, then run a training loop for 5 iterations: compute prediction=W*x, loss=(prediction-y_target)**2, backward, update W, zero grad.](#Set-`learning_rate=0.01`,-then-run-a-training-loop-for-5-iterations:-compute-prediction=W*x,-loss=(prediction-y_target)**2,-backward,-update-W,-zero-grad.)
    - [After the loop, compute the final prediction and print it.](#After-the-loop,-compute-the-final-prediction-and-print-it.)
    - [Print the final value of `W` to see how it changed.](#Print-the-final-value-of-`W`-to-see-how-it-changed.)
- [References](#References)

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# Setup

Before we jump in, let's quickly run the setup cell. It checks the environment (GPU, library versions) and sets random seeds for reproducibility — we want identical results on every run.

In [ ]:
# 🔧 Setup: Run this cell first!
# Environment check + random seeds for reproducibility

import sys
import time
import random
import numpy as np
import pandas as pd
import matplotlib as mpl
import torch
import torch.nn as nn

print(f"📦 Python       {sys.version.split()[0]}")
print(f"🔢 NumPy        {np.__version__}")
print(f"🐼 Pandas       {pd.__version__}")
print(f"📊 Matplotlib   {mpl.__version__}")
print(f"🔥 PyTorch      {torch.__version__}")

# GPU check (only needed for optional deep-feature experiments)
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    device = torch.device("cpu")
    print("⚠️ No GPU detected")

DEVICE_LABEL = "GPU" if device.type == "cuda" else "CPU"

# Random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"\n🎲 Random seed set to {SEED} (Python, NumPy, PyTorch)")

%matplotlib inline
%config InlineBackend.figure_format = 'jpeg'   # photos as JPEG, not PNG — ~10× smaller notebook
# Remove character limits for all columns of Pandas dataframes permanently in this session
pd.set_option('display.max_colwidth', None)

📦 Python       3.13.15
🔢 NumPy        2.1.3
🐼 Pandas       2.2.3
📊 Matplotlib   3.10.0
🔥 PyTorch      2.11.0+cpu
⚠️ No GPU detected

🎲 Random seed set to 42 (Python, NumPy, PyTorch)


![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# PyTorch Tensors: Creation and Basics

PyTorch tensors are the foundation of deep learning in PyTorch. They're similar to NumPy arrays but can run on GPUs and support automatic differentiation.

**Creating tensors**:

```python
import torch

# From Python list
tensor = torch.tensor([1, 2, 3, 4, 5])

# From NumPy array
import numpy as np
np_arr = np.array([1, 2, 3])
tensor = torch.from_numpy(np_arr)

# Zeros and ones
zeros = torch.zeros(3, 4)  # 3x4 tensor of zeros
ones = torch.ones(2, 3)    # 2x3 tensor of ones

# Random tensors
rand_tensor = torch.randn(3, 3)  # Random normal distribution
rand_uniform = torch.rand(2, 2)  # Uniform [0, 1)
```

**Tensor attributes**:

```python
tensor = torch.tensor([[1, 2], [3, 4]])
print(tensor.shape)   # torch.Size([2, 2])
print(tensor.dtype)   # torch.int64
print(tensor.device)  # cpu (or cuda if on GPU)
```

**NumPy conversion**:

```python
# Tensor to NumPy (shares memory!)
np_arr = tensor.numpy()

# NumPy to Tensor
tensor = torch.from_numpy(np_arr)
```

**GPU support**:

```python
# Check if CUDA is available
if torch.cuda.is_available():
    tensor = tensor.to('cuda')  # Move to GPU
    tensor = tensor.to('cpu')   # Move back to CPU

# Create directly on GPU
tensor = torch.randn(3, 3, device='cuda')
```

**Why PyTorch tensors?**

* GPU acceleration (100x+ faster for large operations)
* Automatic differentiation (core of deep learning)
* Integrated with PyTorch's neural network library
* Dynamic computation graphs (flexible model building)

Tensors are the data structure you'll use for everything in deep learning: inputs, outputs, weights, gradients, and more.

## Exercises

### Import torch

In [1]:
import torch

### Create a tensor from the list `[1.0, 2.0, 3.0, 4.0]` and store it in `tensor`.

In [2]:
tensor = torch.tensor([1.0, 2.0, 3.0, 4.0])
print(tensor)

tensor([1., 2., 3., 4.])


### Create a 2x3 tensor of zeros.

In [3]:
zeros = torch.zeros(2, 3)
print(zeros)

tensor([[0., 0., 0.],
        [0., 0., 0.]])


### Create a 3x3 tensor with random values from a normal distribution.

In [4]:
rand_normal = torch.randn(3, 3)
print(rand_normal)

tensor([[-0.9666, -0.0655, -0.4880],
        [ 0.3950, -0.9507,  0.4802],
        [-0.6728, -2.3086, -1.8692]])


### Print the shape, dtype, and device of the `tensor` variable.

In [5]:
print(tensor.shape)
print(tensor.dtype)
print(tensor.device)

torch.Size([4])
torch.float32
cpu


![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# PyTorch: Tensor Operations

PyTorch tensor operations are similar to NumPy but designed for GPU acceleration and automatic differentiation.

**Element-wise operations**:

```python
tensor = torch.tensor([1.0, 2.0, 3.0])
print(tensor + 10)  # [11., 12., 13.]
print(tensor * 2)   # [2., 4., 6.]
print(tensor ** 2)  # [1., 4., 9.]
```

**Matrix operations**:

```python
A = torch.randn(3, 4)
B = torch.randn(4, 5)

# Matrix multiplication
C = A @ B          # Shape: (3, 5)
C = torch.matmul(A, B)  # Same thing

# Element-wise multiplication (different!)
D = A * A  # Must be same shape
```

**Reshaping**:

```python
tensor = torch.randn(12)

# Reshape (creates new view if possible)
reshaped = tensor.reshape(3, 4)
reshaped = tensor.view(3, 4)  # Same, but requires contiguous memory

# Transpose
matrix = torch.randn(3, 4)
transposed = matrix.T  # Shape: (4, 3)
```

**Indexing (like NumPy)**:

```python
tensor = torch.randn(3, 4)
print(tensor[0])      # First row
print(tensor[:, 1])   # Second column
print(tensor[1:, :2]) # Rows 1+, first 2 cols
```

**Reduction operations**:

```python
tensor = torch.randn(3, 4)
print(tensor.sum())        # Sum all elements
print(tensor.mean())       # Average
print(tensor.sum(dim=0))   # Sum along dimension 0
print(tensor.max(dim=1))   # Max along dimension 1
```

**In-place operations**:

```python
tensor = torch.ones(3, 3)
tensor.add_(5)     # In-place add (modifies tensor)
tensor.mul_(2)     # In-place multiply
# Operations ending with _ are in-place
```

**Why these operations matter**:

Every neural network operation (linear layers, convolutions, activations) is built from these primitives. Understanding tensor operations is essential for deep learning.

## Exercises

### Import torch and create tensor `[2.0, 4.0, 6.0, 8.0]`.

In [6]:
import torch

tensor = torch.tensor([2.0, 4.0, 6.0, 8.0])
print(tensor)

tensor([2., 4., 6., 8.])


### Square each element and store in `squared`.

In [7]:
squared = tensor ** 2
print(squared)

tensor([ 4., 16., 36., 64.])


### Create a 3x4 random tensor, then reshape it to 4x3 using `.view()`.

In [8]:
tensor = torch.randn(3, 4)
reshaped = tensor.view(4, 3)
print(reshaped)

tensor([[-0.8271, -0.0514,  0.2312],
        [-0.0356, -0.5972, -1.8537],
        [ 0.3746, -0.0439,  0.2714],
        [-0.1057,  1.5380,  2.1814]])


### Create two tensors of shape (2,3) and (3,4), then multiply them using `@`.

In [9]:
A = torch.randn(2, 3)
B = torch.randn(3, 4)
C = A @ B
print(C)

tensor([[ 0.6197,  0.1795, -0.0898, -0.2783],
        [ 0.9018,  0.6364, -0.4684, -0.7725]])


### Compute the mean and sum of tensor `C`.

In [10]:
print(C.mean())
print(C.sum())

tensor(0.0910)
tensor(0.7284)


![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# PyTorch: Autograd Basics

Automatic differentiation (autograd) is PyTorch's most powerful feature. It automatically computes gradients, which are essential for backpropagation - the algorithm that trains neural networks. For now, just understand that computing gradients is critical for backpropagation to work.

**The `requires_grad` flag**:

```python
x = torch.tensor([2.0], requires_grad=True)
y = x ** 2
print(y)  # tensor([4.], grad_fn=<PowBackward0>)
```

When `requires_grad=True`, PyTorch tracks all operations on this tensor so it can compute gradients later.

**Computing gradients**:

```python
x = torch.tensor([3.0], requires_grad=True)
y = x ** 2 + 2 * x + 1

# Compute gradient of y with respect to x
y.backward()  # Computes dy/dx
print(x.grad)  # tensor([8.]) = 2*3 + 2
```

The `.backward()` method computes the derivative automatically - no manual calculus needed!

**Multiple operations**:

```python
a = torch.tensor([2.0], requires_grad=True)
b = torch.tensor([3.0], requires_grad=True)
c = a * b
d = c + a ** 2

d.backward()
print(a.grad)  # ∂d/∂a
print(b.grad)  # ∂d/∂b
```

PyTorch computes gradients for all tensors in the computation graph.

**Zeroing gradients (important!)**:

```python
# Gradients accumulate by default!
x.grad.zero_()  # Reset gradients to zero
# Or safely:
if x.grad is not None:
    x.grad.zero_()
```

**Why this matters for neural networks**:

Training a neural network follows this pattern:

* **Forward pass**: compute predictions → `y = model(x)`
* **Compute loss**: measure error → `loss = criterion(y, target)`
* **Backward pass**: compute gradients → `loss.backward()`
* **Update weights**: improve model → `optimizer.step()`

Without autograd, you'd have to manually compute derivatives for every operation - incredibly tedious and error-prone! PyTorch handles all of this automatically by building a dynamic computation graph that tracks operations and applies the chain rule during `.backward()`.

## Exercises

### Import torch and create a tensor with value 3.0 and `requires_grad=True`.

In [11]:
import torch

x = torch.tensor([3.0], requires_grad=True)
print(x)

tensor([3.], requires_grad=True)


### Compute `y = x**2 + 2x + 1`.

In [12]:
y = x ** 2 + 2 * x + 1
print(y)

tensor([16.], grad_fn=<AddBackward0>)


### Call `backward()` on `y` to compute gradients.

In [13]:
y.backward()
print(y)

tensor([16.], grad_fn=<AddBackward0>)


### Print `x.grad` to see the gradient dy/dx.

In [14]:
print(x.grad)

tensor([8.])


### Create two tensors `a=2.0` and `b=3.0` with `requires_grad=True`, compute `c = a*b + a**2`, call backward, and print both gradients.

In [15]:
a, b = torch.tensor([2.0], requires_grad=True), torch.tensor([3.0], requires_grad=True)
c = a * b + a ** 2
c.backward()
print(a.grad)
print(b.grad)

tensor([7.])
tensor([2.])


![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# PyTorch: Training Loop Pattern

Now let's put it all together! Every neural network training loop follows the same pattern. Understanding this pattern is crucial before diving into building actual neural networks.

**The standard training loop**:

```python
# 1. Create model parameters (weights)
W = torch.randn(10, 5, requires_grad=True)
b = torch.zeros(5, requires_grad=True)

# 2. Training loop
for epoch in range(100):
    # Forward pass: compute predictions
    predictions = X @ W + b
    
    # Compute loss
    loss = ((predictions - y_target) ** 2).mean()
    
    # Backward pass: compute gradients
    loss.backward()
    
    # Update weights (gradient descent)
    with torch.no_grad():
        W -= learning_rate * W.grad
        b -= learning_rate * b.grad
    
    # Zero gradients for next iteration
    W.grad.zero_()
    b.grad.zero_()
```

**Key components**:

* **Forward pass**: Compute predictions from inputs
* **Loss computation**: Measure how wrong predictions are
* **Backward pass (`.backward()`)**: Compute gradients
* **Parameter update**: Adjust weights using gradients
* **Zero gradients**: Reset for next iteration (gradients accumulate!)

**Why `torch.no_grad()`?**

```python
with torch.no_grad():
    W -= lr * W.grad
```

This disables gradient tracking during weight updates. We don't need gradients of the update operation itself, only of the forward pass.

**This pattern scales**:

Whether you're training a simple linear model or GPT-4, the loop structure is the same:

* Forward → Loss → Backward → Update → Repeat

In Micrograd lesson, you'll implement this from scratch. In PyTorch, you'll use optimizers that handle the update step:

```python
optimizer = torch.optim.SGD([W, b], lr=0.01)
optimizer.zero_grad()  # Instead of manual zeroing
loss.backward()
optimizer.step()       # Instead of manual update
```

But the core pattern remains the same!

## Exercises

### Import torch and create a weight tensor `W` with value 0.5 and `requires_grad=True`.

In [16]:
import torch

W = torch.tensor([0.5], requires_grad=True)
print(W)

tensor([0.5000], requires_grad=True)


### Create input `x=2.0` and target `y_target=10.0` (no requires_grad needed for these).

In [17]:
x = torch.tensor([2.0])
y_target = torch.tensor([10.0])
print(x)
print(y_target)

tensor([2.])
tensor([10.])


### Set `learning_rate=0.01`, then run a training loop for 5 iterations: compute prediction=W*x, loss=(prediction-y_target)**2, backward, update W, zero grad.

In [18]:
learning_rate = 0.01

for epoch in range(5):
    prediction = W * x  # Compute prediction
    loss = (prediction - y_target) ** 2  # Compute loss
    loss.backward()
    with torch.no_grad():
        W -= learning_rate * W.grad  # Update W
    W.grad.zero_()  # Zero grad

### After the loop, compute the final prediction and print it.

In [19]:
final_prediction = W * x
print(final_prediction)

tensor([4.0683], grad_fn=<MulBackward0>)


### Print the final value of `W` to see how it changed.

In [20]:
print(W)

tensor([2.0341], requires_grad=True)


![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)

# References

- **Andrej Karpathy's Zero to Hero Course**

![rainbow](https://raw.githubusercontent.com/ancilcleetus/AI-Learning-Lab/main/assets/rainbow-divider.png)